## config

In [1]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MultiLabelBinarizer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
import matplotlib.pyplot as plt
import re

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import PCA

import json
import glob



In [2]:
pd.set_option('display.float_format', '{:.4f}'.format)

## pp

In [16]:
all_data = []
for file in glob.glob("../data/processed/*.json"):
    print(file)
    with open(file, "r") as f:
        all_data.extend(json.load(f))

df = pd.DataFrame(all_data)
df = df.dropna(thresh=8)
df = df.reset_index(drop=True)

../data/processed\channels0_99.json
../data/processed\channels100_.json
../data/processed\channels3425_.json
../data/processed\channels6727_.json
../data/processed\channelsa10034_.json
../data/processed\channelsa13313_.json
../data/processed\channels_missing.json


### only english chars

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15743 entries, 0 to 15742
Data columns (total 13 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   channel_id                   15743 non-null  object 
 1   channel_name                 15743 non-null  object 
 2   description                  15743 non-null  object 
 3   country                      13977 non-null  object 
 4   defaultLanguage              1725 non-null   object 
 5   created_date                 15743 non-null  object 
 6   category                     15743 non-null  object 
 7   aggregated_tags              15743 non-null  object 
 8   most_common_video_genre      15743 non-null  object 
 9   all_video_genres             15743 non-null  object 
 10  avg_duration_seconds         15743 non-null  float64
 11  avg_seconds_between_uploads  15708 non-null  float64
 12  recent_video_titles          15743 non-null  object 
dtypes: float64(2), o

In [18]:
def is_english(text):
    if not isinstance(text, str):
        return True
    # Allow ASCII + common special chars + emojis
    return bool(re.match(r'^[\x00-\x7F\u00C0-\u024F\u2000-\u206F\u2100-\u214F\u2190-\u21FF\u2600-\u27BF\U0001F300-\U0001FAFF©®™—…•""'']*$', text))

In [19]:
mask = (
    df["description"].apply(is_english) &
    df["aggregated_tags"].apply(is_english) &
    df["recent_video_titles"].apply(is_english) 
)

df = df[mask]

In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13759 entries, 1 to 15742
Data columns (total 13 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   channel_id                   13759 non-null  object 
 1   channel_name                 13759 non-null  object 
 2   description                  13759 non-null  object 
 3   country                      12130 non-null  object 
 4   defaultLanguage              1423 non-null   object 
 5   created_date                 13759 non-null  object 
 6   category                     13759 non-null  object 
 7   aggregated_tags              13759 non-null  object 
 8   most_common_video_genre      13759 non-null  object 
 9   all_video_genres             13759 non-null  object 
 10  avg_duration_seconds         13759 non-null  float64
 11  avg_seconds_between_uploads  13727 non-null  float64
 12  recent_video_titles          13759 non-null  object 
dtypes: float64(2), object

### other pp

In [21]:
df["created_date"] = pd.to_datetime(df["created_date"], format="ISO8601")

TODO: investigate below scaling step

In [22]:
df["avg_seconds_between_uploads"] = df["avg_seconds_between_uploads"].fillna(df["avg_seconds_between_uploads"].max())

## model building

In [23]:
time_cols = ["created_date"]
num_cols = ["avg_duration_seconds", "avg_seconds_between_uploads"]
basic_cat_cols = ["country", "most_common_video_genre"]
multi_cat_cols = ["category", "all_video_genres"]
text_cols = ["description", "aggregated_tags", "recent_video_titles"]
drop_cols = ["channel_id", "channel_name", "defaultLanguage"]
set(time_cols + num_cols + basic_cat_cols + multi_cat_cols + text_cols + drop_cols) == set(df.columns)

True

In [24]:
class DateTimeToPosix(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X, y=None):
        return (pd.to_datetime(X.iloc[:, 0]).astype(int) // 10**9).values.reshape(-1, 1)
    def get_feature_names_out(self, input_features=None):
        return ["created_date_posix"]

In [25]:
# MLB does not have fit/transform. must wrap in custom transformer, give it fit/transform, and col transformer will accept
class MultiLabelBinarizerTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.mlb = MultiLabelBinarizer()
    
    def fit(self, X, y=None):
        self.mlb.fit(X.iloc[:, 0])
        return self
    
    def transform(self, X, y=None):
        return self.mlb.transform(X.iloc[:, 0])
    
    def get_feature_names_out(self, input_features=None):
        return self.mlb.classes_


### text embedding

In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13759 entries, 1 to 15742
Data columns (total 13 columns):
 #   Column                       Non-Null Count  Dtype              
---  ------                       --------------  -----              
 0   channel_id                   13759 non-null  object             
 1   channel_name                 13759 non-null  object             
 2   description                  13759 non-null  object             
 3   country                      12130 non-null  object             
 4   defaultLanguage              1423 non-null   object             
 5   created_date                 13759 non-null  datetime64[ns, UTC]
 6   category                     13759 non-null  object             
 7   aggregated_tags              13759 non-null  object             
 8   most_common_video_genre      13759 non-null  object             
 9   all_video_genres             13759 non-null  object             
 10  avg_duration_seconds         13759 non-null  float6

In [27]:
df.head()

,channel_id,channel_name,description,country,defaultLanguage,created_date,category,aggregated_tags,most_common_video_genre,all_video_genres,avg_duration_seconds,avg_seconds_between_uploads,recent_video_titles
1,UC3IZKseVpdzPSBaWxBxundA,HYBE LABELS,Welcome to the official YouTube channel of HYB...,KR,None,2008-06-04 08:23:22+00:00,"[Pop music, Music, Music of Asia]","[하이브, 하이브레이블즈, HYBE LABELS, HYBE]",Music,[Music],89.6000,131588.1100,[SANTOS BRAVOS “KAWASAKI (&TEAM Remix)” Lyric ...
2,UCF1JIbMUs6uqoZEY1Haw0GQ,Shemaroo,"Welcome to ShemarooEnt, one of the finest dest...",IN,None,2007-09-01 11:44:51+00:00,"[Film, Entertainment]","[salman khan movies, ramcharana moves, Mega Po...",Entertainment,[Entertainment],5336.1000,45200.0000,[Mega Power Star Ram Charan 👑 | Zanjeer (4K Ac...
3,UCYiGq8XF7YQD00x7wAd62Zg,JuegaGerman,Lento pero seguro.,CL,None,2013-05-19 00:09:13+00:00,"[Action game, Video game culture, Action-adven...","[revenia, juega german, juego de miedo, click ...",Gaming,[Gaming],2046.9000,280466.0000,"[Fotos Tomadas En El Momento PERFECTO 📸, Traba..."
4,UC4NALVCmcmL5ntpV0thoH6w,LooLoo Kids - Nursery Rhymes and Children's Songs,LooLoo Kids💖 is an educational YouTube channel...,US,en,2014-08-05 20:15:33+00:00,"[Entertainment, Music, Film]","[kids videos, children songs, farm song nurser...",Music,[Music],148.5000,181623.8900,[Old Macdonald Had a Farm Song + Johny Johny Y...
5,UCJrDMFOdv1I2k8n9oK_V21w,Tips Official,The proud history of Tips Music Limited (Forme...,IN,None,2007-05-22 10:13:28+00:00,"[Film, Music, Music of Asia, Pop music]","[Itna Main Chahoon Tujhe Koi Kisi Ko Na Chahe,...",Music,[Music],2059.1000,16523.7800,[90's Evergreen Songs | 90's Blockbuster Songs...
